# HySpecNet CAE baseline actual bitstream evaluation

This notebook evaluates CAE / baseline checkpoints on the HySpecNet-11k `easy/test` split using the repository's real `compress/decompress` path in `scripts/evaluate.py`.

It is intended for direct comparison against the hierarchical Mamba K=4 + spatial, RD lambda 0.0003 result. The default table includes the archived full-test Mamba reference row and evaluates any available CAE checkpoints found on Drive or in W&B artifacts.

Metric note: `actual_bpppc` here is the measured entropy-coded payload from model `strings`, averaged by `scripts/evaluate.py` over evaluation batches. It does not include extra serialization of Python-side metadata such as tensor shape unless the model itself puts that metadata into the byte strings.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/master-thesis-code')

# Mount the Google account that contains the gdrive2 mirror. The expected archive already
# exists at gdrive2:/hsi/data/archives/ in the local rclone view.
DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
DRIVE_CHECKPOINTS = DRIVE_HSI / 'checkpoints'
DATA_ARCHIVE = DRIVE_HSI / 'data/archives/hyspecnet_easy_test_data_npy_2026-06-13.tar.zst'
DATA_PARENT = Path('/content/hsi_data')
DATASET_ROOT = DATA_PARENT / 'hyspecnet_easy_test_data_npy'

ARTIFACT_ROOT = DRIVE_HSI / 'remote_artifacts/hyspecnet_cae_actual_bpppc_2026-06-16'

# Full test is EVAL_SUBSET_SIZE = 0. Use a small value like 16 only for smoke testing.
# BATCH_SIZE is the default; memory-heavy 3D baselines override it per spec below.
EVAL_SUBSET_SIZE = 0
BATCH_SIZE = 16
NUM_WORKERS = 2

# W&B is optional. If enabled, the notebook searches output artifacts of recent runs and
# downloads model artifacts whose run metadata matches the baseline names below.
USE_WANDB_ARTIFACTS = True
WANDB_PROJECT = 'hsi-compression-paper'
WANDB_ENTITY = None  # set explicitly if automatic entity detection fails
WANDB_RUN_LIMIT = 500
WANDB_DOWNLOAD_DIR = ARTIFACT_ROOT / 'wandb_artifacts'

# The Mamba row below is the archived full easy/test result from the thesis run.
# Set RUN_OPTIONAL_MAMBA_REEVAL = True only if you want to install mamba-ssm and re-run it.
INCLUDE_ARCHIVED_MAMBA_REFERENCE = True
RUN_OPTIONAL_MAMBA_REEVAL = False

CHECKPOINT_SPECS = [
    {
        'slug': 'baseline_1d_pixel_recon',
        'label': '1D pixel baseline',
        'family': 'active_baseline',
        'required': False,
        'requires_mamba': False,
        'batch_size': 2,
        'candidates': [
            DRIVE_CHECKPOINTS / 'internal_baseline_1d_pixel_ae_latent16_best.pt',
        ],
        'wandb_terms': [
            'baseline_1d_pixel_ae_latent16',
            'baseline_1d_pixel_recon',
            'internal_baseline_1d_pixel_ae',
        ],
    },
    {
        'slug': 'baseline_2d_patch_lic_recon',
        'label': '2D patch LIC baseline',
        'family': 'active_baseline',
        'required': True,
        'requires_mamba': False,
        'batch_size': 16,
        'candidates': [
            DRIVE_CHECKPOINTS / 'internal_baseline_2d_patch_ae_lic_recon_latent16_best.pt',
            DRIVE_CHECKPOINTS / 'internal_baseline_2d_patch_ae_lic_latent16_best.pt',
        ],
        'wandb_terms': [
            'baseline_2d_patch_ae_lic_recon_latent16',
            'baseline_2d_patch_ae_lic',
            'internal_baseline_2d_patch_ae_lic',
        ],
    },
    {
        'slug': 'baseline_3d_patch_recon',
        'label': '3D patch baseline',
        'family': 'active_baseline',
        'required': True,
        'requires_mamba': False,
        'batch_size': 1,
        'candidates': [
            DRIVE_CHECKPOINTS / 'internal_baseline_3d_patch_ae_recon_latent16_best.pt',
        ],
        'wandb_terms': [
            'baseline_3d_patch_ae_recon_latent16',
            'baseline_3d_patch_recon',
            'internal_baseline_3d_patch_ae_recon',
        ],
    },
    {
        'slug': 'baseline_3d_patch_rd_0_01',
        'label': '3D patch RD baseline',
        'family': 'active_baseline',
        'required': True,
        'requires_mamba': False,
        'batch_size': 1,
        'candidates': [
            DRIVE_CHECKPOINTS / 'internal_baseline_3d_patch_ae_latent16_best.pt',
        ],
        'wandb_terms': [
            'baseline_3d_patch_ae_latent16',
            'baseline_3d_patch_rd',
            'internal_baseline_3d_patch_ae',
        ],
    },
    {
        'slug': 'hybrid_2d3d_lic_recon',
        'label': 'Hybrid 2D/3D LIC baseline',
        'family': 'active_baseline',
        'required': False,
        'requires_mamba': False,
        'batch_size': 1,
        'candidates': [
            DRIVE_CHECKPOINTS / 'internal_hybrid_2d3d_ae_lic_recon_latent16_best.pt',
        ],
        'wandb_terms': [
            'hybrid_2d3d_ae_lic_recon_latent16',
            'hybrid_2d3d_lic_recon',
            'internal_hybrid_2d3d_ae_lic',
        ],
    },
]

if RUN_OPTIONAL_MAMBA_REEVAL:
    CHECKPOINT_SPECS.append(
        {
            'slug': 'hierarchical_mamba_k4_rd_0_0003',
            'label': 'Hierarchical Mamba K4 RD 3e-4',
            'family': 'mamba',
            'required': False,
            'requires_mamba': True,
            'candidates': [
                DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_01_best.pt',
                DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_0003_best.pt',
            ],
            'wandb_terms': ['hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_0003'],
        }
    )

print('Drive paths configured. Directories will be created after Drive is mounted.')


## Mount Drive and prepare the repository

In [ ]:
from google.colab import drive

def _path_has_entries(path):
    return path.exists() and any(path.iterdir())

def _mount_drive():
    last_exc = None
    for mountpoint in [Path('/content/drive'), Path('/content/gdrive'), Path('/content/google_drive')]:
        try:
            drive.mount(str(mountpoint), force_remount=False)
            return mountpoint
        except ValueError as exc:
            last_exc = exc
            if 'already contain files' in str(exc):
                print(f'{mountpoint} is non-empty before mount; trying another mountpoint.')
                continue
            raise
    raise last_exc

OLD_DRIVE_HSI = DRIVE_HSI
DRIVE_MOUNT = _mount_drive()
DRIVE_HSI = DRIVE_MOUNT / 'MyDrive/hsi'
DRIVE_CHECKPOINTS = DRIVE_HSI / 'checkpoints'
DATA_ARCHIVE = DRIVE_HSI / 'data/archives/hyspecnet_easy_test_data_npy_2026-06-13.tar.zst'
ARTIFACT_ROOT = DRIVE_HSI / 'remote_artifacts/hyspecnet_cae_actual_bpppc_2026-06-16'
WANDB_DOWNLOAD_DIR = ARTIFACT_ROOT / 'wandb_artifacts'

def _remap_drive_candidate(candidate):
    path = Path(candidate)
    try:
        rel = path.relative_to(OLD_DRIVE_HSI)
    except ValueError:
        return path
    return DRIVE_HSI / rel

for spec in CHECKPOINT_SPECS:
    spec['candidates'] = [_remap_drive_candidate(path) for path in spec.get('candidates', [])]

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
WANDB_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print('Drive mount:', DRIVE_MOUNT)
print('Drive HSI root:', DRIVE_HSI)
print('Artifact root:', ARTIFACT_ROOT)


In [ ]:
import os
import subprocess
import sys

def run(cmd, cwd=None, check=True):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, check=check)

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

os.chdir(REPO_DIR)
print('Repo:', Path.cwd())
run(['git', 'rev-parse', '--short', 'HEAD'])


## Install dependencies

The default CAE path does not require `mamba-ssm`. If `RUN_OPTIONAL_MAMBA_REEVAL=True`, install the same Torch/Mamba binary stack used by the other thesis Colab notebooks before running evaluation.

In [ ]:
import importlib.util

run(['apt-get', 'update'])
run(['apt-get', 'install', '-y', 'zstd'])

PIP = [sys.executable, '-m', 'pip']
run(PIP + ['install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run(PIP + ['install', '-q', '-e', '.', 'pandas', 'tabulate', 'zstandard', 'wandb'])

if RUN_OPTIONAL_MAMBA_REEVAL:
    marker = Path('/content/.hsi_cae_eval_torch27_mamba232')
    if not marker.exists():
        run(PIP + [
            'install', '-q', '--force-reinstall',
            'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
            '--index-url', 'https://download.pytorch.org/whl/cu126',
        ])
        run(PIP + ['install', '-q', '--upgrade', 'packaging', 'pybind11', 'ninja'])
        run(PIP + [
            'install', '-q', '--no-deps', '--force-reinstall',
            'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        ])
        run(PIP + [
            'install', '-q', '--no-deps', '--force-reinstall',
            'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        ])
        marker.write_text('installed\n')
        print('Mamba stack installed. Restart runtime, reconnect, and rerun from the settings cell.')
        os.kill(os.getpid(), 9)

import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())
print('wandb installed:', importlib.util.find_spec('wandb') is not None)
if RUN_OPTIONAL_MAMBA_REEVAL:
    from mamba_ssm import Mamba  # noqa: F401
    print('mamba-ssm import: ok')


## Extract HySpecNet easy/test DATA.npy archive

In [ ]:
import csv
import shutil

if not DATA_ARCHIVE.exists():
    raise FileNotFoundError(
        f'Missing archive: {DATA_ARCHIVE}. Copy it from gdrive2:/hsi/data/archives or edit DATA_ARCHIVE.'
    )

split_csv = DATASET_ROOT / 'splits/easy/test.csv'
if not split_csv.exists():
    DATA_PARENT.mkdir(parents=True, exist_ok=True)
    run(['tar', '--zstd', '-xf', str(DATA_ARCHIVE), '-C', str(DATA_PARENT)])

if not split_csv.exists():
    raise FileNotFoundError(f'Extracted dataset is missing {split_csv}')

with split_csv.open('r', encoding='utf-8') as handle:
    rows = list(csv.reader(handle))
num_rows = len(rows) - 1 if rows and rows[0] and 'path' in rows[0][0].lower() else len(rows)
print('Dataset root:', DATASET_ROOT)
print('Split CSV:', split_csv)
print('easy/test rows:', num_rows)
if num_rows != 1149:
    print('WARNING: expected 1149 easy/test samples for the active HySpecNet protocol.')


## Optional W&B artifact discovery

If this Colab runtime has `WANDB_API_KEY` set, or if you run `wandb.login()`, the cell below searches recent runs for model artifacts matching the baseline names. It downloads `best` model artifacts and adds their `.pt` files as checkpoint candidates.

In [ ]:
import json
from collections import defaultdict

downloaded_from_wandb = defaultdict(list)

def _flatten_text(value):
    try:
        return json.dumps(value, sort_keys=True, default=str).lower()
    except TypeError:
        return str(value).lower()

def _candidate_entities(api):
    entities = []
    if WANDB_ENTITY:
        entities.append(WANDB_ENTITY)
    try:
        viewer = api.viewer
        if getattr(viewer, 'entity', None):
            entities.append(viewer.entity)
        for team in getattr(viewer, 'teams', []):
            name = getattr(team, 'name', None)
            if name:
                entities.append(name)
    except Exception as exc:
        print('Could not inspect W&B viewer/team list:', repr(exc))
    deduped = []
    for entity in entities:
        if entity and entity not in deduped:
            deduped.append(entity)
    return deduped

def _find_project_path(api):
    for entity in _candidate_entities(api):
        try:
            api.project(WANDB_PROJECT, entity=entity)
            return f'{entity}/{WANDB_PROJECT}'
        except Exception:
            pass
    if WANDB_ENTITY:
        return f'{WANDB_ENTITY}/{WANDB_PROJECT}'
    return None

def _artifact_has_best_alias(artifact):
    aliases = [str(alias) for alias in getattr(artifact, 'aliases', [])]
    return 'best' in aliases or not aliases

def _download_artifact_pt(artifact, slug):
    target = WANDB_DOWNLOAD_DIR / slug / artifact.name.replace(':', '_').replace('/', '_')
    path = Path(artifact.download(root=str(target)))
    pt_files = sorted(path.rglob('*.pt'))
    if not pt_files:
        print('No .pt files in artifact:', artifact.name)
        return None
    return pt_files[0]

if USE_WANDB_ARTIFACTS:
    import wandb
    if not os.environ.get('WANDB_API_KEY'):
        print('WANDB_API_KEY is not set. Run wandb.login() in this runtime if artifacts are private.')
    try:
        api = wandb.Api(timeout=60)
        project_path = _find_project_path(api)
        if project_path is None:
            print('W&B project not found. Set WANDB_ENTITY explicitly and rerun this cell.')
        else:
            print('Searching W&B project:', project_path)
            runs = api.runs(project_path, order='-created_at', per_page=50)
            checked = 0
            for run_obj in runs:
                checked += 1
                if checked > WANDB_RUN_LIMIT:
                    break
                run_text = ' '.join(
                    [
                        str(getattr(run_obj, 'id', '')),
                        str(getattr(run_obj, 'name', '')),
                        str(getattr(run_obj, 'display_name', '')),
                        _flatten_text(getattr(run_obj, 'config', {})),
                        _flatten_text(getattr(run_obj, 'summary', {})),
                        _flatten_text(getattr(run_obj, 'tags', [])),
                    ]
                ).lower()
                matched_specs = []
                for spec in CHECKPOINT_SPECS:
                    terms = [term.lower() for term in spec.get('wandb_terms', [])]
                    if any(term in run_text for term in terms):
                        matched_specs.append(spec)
                if not matched_specs:
                    continue
                for artifact in run_obj.logged_artifacts():
                    if getattr(artifact, 'type', None) != 'model':
                        continue
                    if not _artifact_has_best_alias(artifact):
                        continue
                    for spec in matched_specs:
                        print('Downloading', artifact.name, 'for', spec['slug'], 'from run', run_obj.id)
                        pt_path = _download_artifact_pt(artifact, spec['slug'])
                        if pt_path is not None:
                            downloaded_from_wandb[spec['slug']].append(pt_path)
                            spec.setdefault('candidates', []).insert(0, pt_path)
            print('Checked W&B runs:', min(checked, WANDB_RUN_LIMIT))
    except Exception as exc:
        print('W&B artifact discovery skipped:', repr(exc))
else:
    print('W&B artifact discovery disabled.')

dict(downloaded_from_wandb)


## Evaluate available checkpoints

In [ ]:
import time

def resolve_checkpoint(spec):
    for candidate in spec.get('candidates', []):
        path = Path(candidate)
        if path.exists():
            return path
    return None

def eval_json_path(run_name):
    return REPO_DIR / 'artifacts/logs' / f'{run_name}.json'

def run_evaluation(spec, checkpoint_path):
    run_name = f"lossy_compare_{spec['slug']}_easy_test"
    if EVAL_SUBSET_SIZE:
        run_name += f'_subset{EVAL_SUBSET_SIZE}'
    eval_batch_size = int(spec.get('batch_size', BATCH_SIZE))
    cmd = [
        sys.executable,
        'scripts/evaluate.py',
        str(checkpoint_path),
        str(DATASET_ROOT),
        '--split',
        'test',
        '--difficulty',
        'easy',
        '--batch-size',
        str(eval_batch_size),
        '--num-workers',
        str(NUM_WORKERS),
        '--run-name',
        run_name,
        '--save-json',
        '--disable-wandb',
        '--no-progress',
    ]
    if EVAL_SUBSET_SIZE:
        cmd.extend(['--subset-size', str(EVAL_SUBSET_SIZE)])
    env = os.environ.copy()
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    start = time.perf_counter()
    completed = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True, env=env)
    elapsed = time.perf_counter() - start
    stdout_path = ARTIFACT_ROOT / f'{run_name}.stdout.txt'
    stderr_path = ARTIFACT_ROOT / f'{run_name}.stderr.txt'
    stdout_path.write_text(completed.stdout, encoding='utf-8')
    stderr_path.write_text(completed.stderr, encoding='utf-8')
    print('\n' + '=' * 80)
    print(spec['label'])
    print('checkpoint:', checkpoint_path)
    print('batch_size:', eval_batch_size)
    print('elapsed_sec:', round(elapsed, 2))
    print(completed.stdout[-4000:])
    if completed.returncode != 0:
        print(completed.stderr[-4000:])
        raise RuntimeError(f"Evaluation failed for {spec['slug']}; stderr saved to {stderr_path}")
    out_path = eval_json_path(run_name)
    if not out_path.exists():
        raise FileNotFoundError(out_path)
    copied = ARTIFACT_ROOT / out_path.name
    shutil.copy2(out_path, copied)
    return copied

eval_records = []
missing_records = []
for spec in CHECKPOINT_SPECS:
    if spec.get('requires_mamba') and not RUN_OPTIONAL_MAMBA_REEVAL:
        missing_records.append({**spec, 'status': 'skipped_mamba_reeval_disabled'})
        continue
    ckpt = resolve_checkpoint(spec)
    if ckpt is None:
        missing_records.append({**spec, 'status': 'missing_checkpoint'})
        continue
    try:
        json_path = run_evaluation(spec, ckpt)
    except Exception as exc:
        print('Evaluation failed but notebook will continue:', spec['slug'], repr(exc))
        missing_records.append(
            {
                **spec,
                'status': 'eval_failed',
                'checkpoint_path': str(ckpt),
                'error': repr(exc),
            }
        )
        continue
    eval_records.append(
        {
            **spec,
            'checkpoint_path': str(ckpt),
            'eval_json': str(json_path),
            'batch_size': int(spec.get('batch_size', BATCH_SIZE)),
        }
    )

print('Evaluated:', len(eval_records))
print('Missing/skipped:', len(missing_records))
for rec in missing_records:
    print('-', rec['slug'], rec['status'])


## Build comparison table

In [ ]:
import pandas as pd

def load_eval_row(record):
    path = Path(record['eval_json'])
    data = json.loads(path.read_text())
    metrics = data.get('metrics', data)
    return {
        'slug': record['slug'],
        'model': record['label'],
        'family': record['family'],
        'status': 'evaluated',
        'source': 'fresh_eval',
        'batch_size': record.get('batch_size'),
        'num_samples': data.get('num_samples'),
        'num_input_bands': data.get('num_input_bands'),
        'actual_bpppc': metrics.get('actual_bpppc'),
        'actual_CR': metrics.get('actual_compression_ratio'),
        'actual_PSNR': metrics.get('actual_psnr', metrics.get('psnr')),
        'actual_SSIM': metrics.get('actual_ssim', metrics.get('ssim')),
        'actual_SA_deg': metrics.get('actual_sam_deg', metrics.get('actual_sa_deg', metrics.get('sa_deg'))),
        'likelihood_bpppc': metrics.get('likelihood_bpppc'),
        'proxy_bpppc': metrics.get('proxy_bpppc'),
        'encode_ms_per_batch': metrics.get('encode_ms_per_batch'),
        'decode_ms_per_batch': metrics.get('decode_ms_per_batch'),
        'checkpoint': record.get('checkpoint_path'),
        'eval_json': record.get('eval_json'),
        'error': None,
    }

rows = [load_eval_row(record) for record in eval_records]

if INCLUDE_ARCHIVED_MAMBA_REFERENCE:
    rows.append(
        {
            'slug': 'hierarchical_mamba_k4_rd_0_0003_archived',
            'model': 'Hierarchical Mamba K4 RD 3e-4',
            'family': 'mamba',
            'status': 'archived_reference',
            'source': 'archived_full_easy_test',
            'batch_size': None,
            'num_samples': 1149,
            'num_input_bands': 202,
            'actual_bpppc': 0.1126870636892791,
            'actual_CR': 141.9861293406141,
            'actual_PSNR': 45.024777253468834,
            'actual_SSIM': 0.9829308769355217,
            'actual_SA_deg': 3.324809943739739,
            'likelihood_bpppc': 0.11262269747546977,
            'proxy_bpppc': 0.029702970297029667,
            'encode_ms_per_batch': 176.2152469400462,
            'decode_ms_per_batch': 69.28469250376818,
            'checkpoint': 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_01_best.pt',
            'eval_json': 'artifacts/analysis/hierarchical_mamba_rd_20260509/eval_json/k4_spatial_rd_lambda_0_0003.json',
            'error': None,
        }
    )

for rec in missing_records:
    rows.append(
        {
            'slug': rec['slug'],
            'model': rec['label'],
            'family': rec['family'],
            'status': rec['status'],
            'source': 'not_evaluated',
            'batch_size': rec.get('batch_size'),
            'num_samples': None,
            'num_input_bands': None,
            'actual_bpppc': None,
            'actual_CR': None,
            'actual_PSNR': None,
            'actual_SSIM': None,
            'actual_SA_deg': None,
            'likelihood_bpppc': None,
            'proxy_bpppc': None,
            'encode_ms_per_batch': None,
            'decode_ms_per_batch': None,
            'checkpoint': rec.get('checkpoint_path') or '; '.join(str(p) for p in rec.get('candidates', [])),
            'eval_json': None,
            'error': rec.get('error'),
        }
    )

df = pd.DataFrame(rows)
sort_cols = ['status', 'actual_bpppc', 'model']
df = df.sort_values(sort_cols, na_position='last').reset_index(drop=True)

csv_path = ARTIFACT_ROOT / 'hyspecnet_cae_actual_bpppc_comparison.csv'
json_path = ARTIFACT_ROOT / 'hyspecnet_cae_actual_bpppc_comparison.json'
md_path = ARTIFACT_ROOT / 'hyspecnet_cae_actual_bpppc_comparison.md'
df.to_csv(csv_path, index=False)
df.to_json(json_path, orient='records', indent=2)

display_cols = [
    'model',
    'family',
    'status',
    'source',
    'batch_size',
    'num_samples',
    'actual_bpppc',
    'actual_CR',
    'actual_PSNR',
    'actual_SSIM',
    'actual_SA_deg',
    'likelihood_bpppc',
    'error',
]
markdown_table = df[display_cols].to_markdown(index=False, floatfmt='.6f')
notes = [
    '# HySpecNet CAE actual bpppc comparison',
    '',
    f'Dataset root: `{DATASET_ROOT}`',
    f'Split: `easy/test`',
    f'EVAL_SUBSET_SIZE: `{EVAL_SUBSET_SIZE}` (`0` means full test)',
    f'Default batch size: `{BATCH_SIZE}`; 3D/hybrid rows use per-model overrides to avoid Colab T4 OOM.',
    '',
    markdown_table,
    '',
    'Notes:',
    '- `actual_bpppc` and `actual_CR` for fresh rows come from `scripts/evaluate.py` real `compress/decompress`.',
    '- `scripts/evaluate.py` averages bpppc over batches; use the same batch size for strict table parity.',
    '- Each evaluation writes stdout/stderr logs next to this report in `ARTIFACT_ROOT`.',
    '- The archived Mamba row is the full easy/test K4 + spatial RD lambda 0.0003 result already used in the thesis artifacts.',
    '- Missing CAE rows need their checkpoint uploaded to `MyDrive/hsi/checkpoints` or available as a W&B model artifact.',
]
md_path.write_text('\n'.join(notes) + '\n', encoding='utf-8')

print('Saved:', csv_path)
print('Saved:', json_path)
print('Saved:', md_path)
df[display_cols]


## Interpretation checklist

Use the table as direct per-checkpoint comparison only when rows are `evaluated` or the row is explicitly the archived Mamba full-test reference. Rows marked `missing_checkpoint` are not evidence about model performance; they only show that the notebook did not find a usable checkpoint in Drive or W&B.

For thesis/reporting text, keep `actual_bpppc` separate from `proxy_bpppc` and `likelihood_bpppc`. The CAE proxy rate is only a latent-size diagnostic; the measured comparison should use `actual_bpppc` from the real byte strings.